In [ ]:
from pathlib import Path

mintpy_timeseries_h5 = Path("/Volumes/Fortress_L3/SnowWaterEquivalent/ALOS2_CA_P67_F750_A/CA_P67_F750_A_Dec22_June23/mintpy/geo/geo_timeseries_ERA5_demErr.h5")  # <- change
wsdlurl = "https://hydroportal.cuahsi.org/Snotel/cuahsi_1_1.asmx?WSDL"

obs_hour = 0
reference_date = "12-01"
include_temperature = True


In [ ]:
from utils.insar_context import build_insar_context

ctx = build_insar_context(
    source="mintpy",
    mintpy_timeseries_h5=mintpy_timeseries_h5,
    mintpy_reference_slice=None,
)

ctx.source, len(ctx.dates), ctx.footprint

In [ ]:
from utils.snotel_utils import fetch_snotel_sites, filter_sites_by_polygon

sites = fetch_snotel_sites(wsdlurl)
snotel_sites = filter_sites_by_polygon(sites, ctx.footprint.iloc[0].geometry)

print("Stations in footprint:", len(snotel_sites))
snotel_sites.head()


In [ ]:
from utils.snotel_utils import fetch_snotel_timeseries

swe_data = fetch_snotel_timeseries(
    snotel_sites,
    wsdlurl,
    start_date=str(ctx.dates[0].date()),
    end_date=str(ctx.dates[-1].date()),
    reference_date=reference_date,
    obs_hour=obs_hour,
    include_temperature=include_temperature,
)

print("Stations returned:", len(swe_data))
if not swe_data:
    raise RuntimeError("No stations succeeded; check site codes and date range.")


In [ ]:
from utils.plotting import make_footprint_station_map

m = make_footprint_station_map(ctx.footprint, snotel_sites, zoom_start=8)
m


In [ ]:
from utils.plotting import plot_snotel_data  # or wherever you placed it

plot_snotel_data(swe_data, [d.date() for d in ctx.dates])


In [ ]:
from utils import save_pickle
from pathlib import Path

Path("cache").mkdir(exist_ok=True)
save_pickle(swe_data, "cache/snotel_data_mintpy.pkl")
